## 导库

In [ ]:
import re
import torch
import torch.nn as nn
import numpy as np
import random
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader, RandomSampler
import torch.nn.functional as F
import torch.optim as optim
import time
import math
import jieba
import plotly.express as px
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### 工具函数

In [2]:
def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)


def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

## 数据预处理

为了不影响准确度，如果字符属于数字，就将其替换为数字特殊 token-
“[num]”，处理后句子变成“[num]年”；

如果字符属于其他语言，就将其替换为字母特殊 token-“[eng]”，处理后句子变成“[eng]”；


In [140]:
filtering_data_file = './data/MuCGEC_dev.txt'
filtered_data_file = './data/MuCGEC_dev_filtered.txt'


with open(filtering_data_file, 'r') as f:
    lines = f.readlines()
    # 去掉最前面的数字和制表符
    sentences = [line.split('\t', 1)[1] for line in lines]
    sentences = [sentence.split('\t') for sentence in sentences]
    eng_pattern = re.compile(r'[a-zA-Z]+')
    num_pattern = re.compile(r'\d+')
    # 去掉最后的换行符
    for i, sentence in enumerate(sentences):
        for j, s in enumerate(sentence):

            s = s.strip('\n').strip('。')
            # 使用sub方法将非中文部分替换为[eng]
            eng_found = eng_pattern.findall(s)
            for eng in eng_found:
                s = s.replace(eng, '[eng]')
            num_found = num_pattern.findall(s)
            for num in num_found:
                s = s.replace(num, '[num]')
            # 将非中文部分的.替换为。
            sentence[j] = s.replace('.', '。').replace('?', '？').replace(
                '!', '！').replace(';', '；').replace(',', '，')

        sentences[i] = sentence
print(sentences[:10])

[['因为在冰箱里没什么东西也做很好吃的菜', '即使在冰箱里没什么东西也能做很好吃的菜', '即使在冰箱里没什么东西，也能做很好吃的菜'], ['再说，他们可能以为自己吸烟时对别人带来的不便和不舒服也不可能很大', '再说，他们可能以为自己吸烟时给别人带来的不便和不舒服也不会很大', '再说，他们可能以为自己吸烟时给别人带来的不便和不舒服也不是很大', '再说，他们可能以为自己吸烟时给别人带来的不便和不舒服也不可能很大'], ['这样我们在生活当中没有了跟男生接触的机会，总会有一天离开学校的，我们那时肯定会遇到到底怎样与男人交流、沟通的问题了', '这样我们在生活当中没有了跟男生接触的机会，但我们总会有一天要离开学校的，我们离开学校后肯定会遇到到底应该怎样与男人交流、沟通的问题', '这样我们在生活当中没有了跟男生接触的机会，等有一天离开学校的时候，我们肯定会遇到到底应该怎样与男人交流、沟通的问题', '这样我们在生活当中没有了跟男生接触的机会，我们总会有一天离开学校的，我们那时肯定会遇到到底怎样与男人交流、沟通的问题'], ['我不愿意这么过分的喜欢歌手', '我不愿意这么过分地喜欢歌手', '我不愿意这么过分痴迷地喜欢歌手'], ['学生大概做飞机去北京', '学生大概坐飞机去北京', '学生大概是坐飞机去北京'], ['今天我看见了我家周边的西药店，吊着牌子称；“口罩都卖光了”', '今天我看见我家周边的西药店，吊着牌子称：“口罩都卖光了。”', '今天我看见我家周边的西药店吊着牌子称：“口罩都卖光了。”'], ['我是中学生，不知因为我是喜爱流行、时尚的人，但我认为流行歌曲是一种自己享受快乐的一种方法', '我是中学生，不知是否是因为我是一个喜爱流行、时尚的人才会这么想，但我确实认为流行歌曲是一种自己享受快乐的方法', '我是中学生，不知是否是因为我是一个喜爱流行、时尚的人才会这么想，但我确实认为流行歌曲是自己享受快乐的一种方法'], ['改变自己的身体的冲动往往是一个深刻精神不稳定的症状。它应该被视为一个问题，而不是纵容还有鼓励美容手术治疗', '改变自己身体的冲动往往是一种深层次的精神不稳定的症状。它应该被视为是一个问题，而不是纵容、鼓励美容手术治疗', '改变自己的身体的冲动往往是一个精神极不稳定的症状。它应该被视为一个问题，而不是被纵容或者被鼓励采

每一个错误句子只能有一个正确句子  
如果一行只有一个句子，认为他是正确的，对应的正确句也就是他  
如果一行有多个句子，认为有多个正确句，遍历正确句，每一个正确句和错误句组成一个样本


In [141]:
filtered_sentences = []
for index, sentence in enumerate(sentences):
    if len(sentence) == 1:
        filtered_sentences.append([sentence[0], sentence[0]])
        continue
    elif len(sentence) == 2:
        if sentence[1] == '没有错误':
            sentence[1] = sentence[0]
        filtered_sentences.append(sentence)
        continue
    elif len(sentence) > 2:
        for i in range(1, len(sentence)):
            filtered_sentences.append([sentence[0], sentence[i]])
        filtered_sentences.pop(index)
    elif sentence[1].find('[缺失成分]') or sentence[1].find('[缺失成分]'):
        filtered_sentences.pop(index)
print(filtered_sentences[:10])

[['因为在冰箱里没什么东西也做很好吃的菜', '即使在冰箱里没什么东西，也能做很好吃的菜'], ['再说，他们可能以为自己吸烟时对别人带来的不便和不舒服也不可能很大', '再说，他们可能以为自己吸烟时给别人带来的不便和不舒服也不是很大'], ['这样我们在生活当中没有了跟男生接触的机会，总会有一天离开学校的，我们那时肯定会遇到到底怎样与男人交流、沟通的问题了', '这样我们在生活当中没有了跟男生接触的机会，但我们总会有一天要离开学校的，我们离开学校后肯定会遇到到底应该怎样与男人交流、沟通的问题'], ['这样我们在生活当中没有了跟男生接触的机会，总会有一天离开学校的，我们那时肯定会遇到到底怎样与男人交流、沟通的问题了', '这样我们在生活当中没有了跟男生接触的机会，我们总会有一天离开学校的，我们那时肯定会遇到到底怎样与男人交流、沟通的问题'], ['我不愿意这么过分的喜欢歌手', '我不愿意这么过分痴迷地喜欢歌手'], ['学生大概做飞机去北京', '学生大概是坐飞机去北京'], ['今天我看见了我家周边的西药店，吊着牌子称；“口罩都卖光了”', '今天我看见我家周边的西药店吊着牌子称：“口罩都卖光了。”'], ['我是中学生，不知因为我是喜爱流行、时尚的人，但我认为流行歌曲是一种自己享受快乐的一种方法', '我是中学生，不知是否是因为我是一个喜爱流行、时尚的人才会这么想，但我确实认为流行歌曲是自己享受快乐的一种方法'], ['改变自己的身体的冲动往往是一个深刻精神不稳定的症状。它应该被视为一个问题，而不是纵容还有鼓励美容手术治疗', '改变自己的身体的冲动往往是一个精神极不稳定的症状。它应该被视为一个问题，而不是被纵容或者被鼓励采用美容手术治疗'], ['这个巨大飞的物体停住了一会就又向天空快速地飞去。一闪而过，变成了一个形影一样的亮点，一点点地暗下去', '这个巨大的飞行物体停了一会儿就又向天空快速地飞去了。它一闪而过，变成了一个星星一样的亮点，然后一点点地暗下去']]


写入文件**filtered.txt中

In [142]:
with open(filtered_data_file, 'w') as f:
    for sentence in filtered_sentences:
        f.write(sentence[0] + ' ' + sentence[1] + '\n')

## 加载数据

In [3]:
train_data_file = './data/MuCGEC_dev_filtered.txt'  # 训练数据
test_data_file = './data/MuCGEC_test_filtered.txt'

### 辅助加载类

In [4]:
SOS_token = 0
EOS_token = 1


class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "[SOS]", 1: "[EOS]"}
        # self.index2word = {}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in jieba.lcut(sentence):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

根据每段句子的长度，确定最大长度

In [10]:
with open(train_data_file, 'r') as f:
    lines = f.readlines()
    sentences = [[s for s in l.split(' ')] for l in lines]
# 计算每个句子的长度
sentence_lengths = [len(jieba.lcut(sentence_pair[0]))
                    for sentence_pair in sentences]

# 使用Plotly绘制直方图
fig = px.histogram(sentence_lengths, nbins=200, title='Sentence Length Distribution', labels={
                   'value': 'Sentence Length', 'count': 'Frequency'})
fig.show()

将数据集修剪为相对较短且简单的句子

In [11]:
MAX_LENGTH = 40


def filterPair(p):
    # 因为后面会加<EOS>，所以这里减去一些长度
    ret = len(jieba.lcut(p[0])) < MAX_LENGTH - 6 and len(
        jieba.lcut(p[0])) < MAX_LENGTH - 6
    return ret

In [12]:
def prepareData(data_file=train_data_file):

    input_lang = Lang('input')
    output_lang = Lang('output')
    pairs = []

    with open(data_file, 'r') as f:
        lines = f.readlines()
        for line in lines:
            pair = line.strip('\n').split(' ')
            if not filterPair(pair):
                # 如果长度不符合要求则跳过
                continue
            pairs.append(pair)
            input_lang.addSentence(pair[0])
            output_lang.addSentence(pair[1])
    return input_lang, output_lang, pairs

### 准备数据

In [13]:
input_lang, output_lang, pairs = prepareData(data_file=train_data_file)
print("Counted words:")
print(input_lang.name, input_lang.n_words)
print(output_lang.name, output_lang.n_words)
print(random.choice(pairs))

Counted words:
input 2883
output 2928
['”这样注意的话，肯定对孩子好的影响', '这样注意的话，肯定对孩子有好的影响']


## 构建模型

### 定义编码器

In [14]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.gru(embedded)
        return output, hidden

### 定义解码器

### 构建注意力机制

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)
        self.Ua = nn.Linear(hidden_size, hidden_size)
        self.Va = nn.Linear(hidden_size, 1)

    def forward(self, query, keys):
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))
        scores = scores.squeeze(2).unsqueeze(1)

        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, keys)

        return context, weights


class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = BahdanauAttention(hidden_size)
        self.gru = nn.GRU(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(
            batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)
            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(
                    1)  # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                # detach from history as input
                decoder_input = topi.squeeze(-1).detach()

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions

    def forward_step(self, input, hidden, encoder_outputs):
        embedded = self.dropout(self.embedding(input))

        query = hidden.permute(1, 0, 2)
        context, attn_weights = self.attention(query, encoder_outputs)
        input_gru = torch.cat((embedded, context), dim=2)

        output, hidden = self.gru(input_gru, hidden)
        output = self.out(output)

        return output, hidden, attn_weights

## 训练模型

In [39]:
def indexesFromSentence(lang, sentence):
    # 将句子中的每个单词转换为对应的索引
    return [lang.word2index[word] for word in jieba.lcut(sentence)]


def tensorFromSentence(lang, sentence):
    # 将句子转换为索引的Tensor
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)


def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)


def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData()

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(
        train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

In [ ]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
                decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(
            encoder_outputs, encoder_hidden, target_tensor)
        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

### 训练

In [41]:
plt.switch_backend('agg')


def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [42]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
          print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder,
                           encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                         epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

## 评估模型

In [43]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(
            encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [44]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(
            encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence.replace(' ', '').replace('<EOS>', ''))
        print('')

### 训练启动

In [49]:
hidden_size = 256
batch_size = 64
n_epochs = 100

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder,
      n_epochs, print_every=1, plot_every=1)

torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([64, 1, 2928])
torch.Size([

KeyboardInterrupt: 

In [37]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> 其次，吸烟给周围人比自己更深刻的影响
= 其次，吸烟给周围人的影响比自己更严重
< 其次，吸烟会影响健康的影响比自己更严重影响

> 这个学校是日本和中国的政府建立的，根据小册子在学校老师一百多，还有学校的管理严格
= 这个学校是日本和中国的政府一起建立的，根据小册子所写，在学校的老师有一百多位，并且学校的管理非常严格
< 这个学校是日本和中国的老师，有一百多位老师，并且学校的管理十分严格

> 吃了又健康
= 好吃又健康
< 好吃又健康

> 孩子的学习能力是非常高的。他们在全取决于父母的教导
= 孩子的学习能力是非常强的。他们全取决于父母的教导
< 他们的是非常强的。他们全取决于父母的教导

> 如果贵公司肯聘请我，信相对你们来说是百利而无一弊的
= 如果贵公司肯聘请我，相信对你们来说是有百利而无一弊的
< 如果贵公司肯聘请我，相信对你们来说是有百利而无一弊的事情

> 军队能让年轻人成熟起来吗？对于我观点有凭证：我觉得是不可能的
= 军队能让年轻人成熟起来吗？对于这点，我观点是：这是不可能的
< 军队能让年轻人成熟起来吗？对于我是不可能的

> 综上所述，我认为当父母极其是个很难的事，为了养成孩子，应该要自己的本身的能力提高
= 综上所述，我认为当父母是个很难的事，为了养成孩子，父母应该要把本身的能力提高
< 综上所述，父母是个极其难的事，为了养成孩子，父母应该要把本身的能力提高

> 文凭不准是成功的保证之一，也是因为一个人的文凭不相当于这个人实际的价值
= 文凭不是成功的保证之一，也是因为一个人的文凭不等于这个人实际的价值
< 文凭不是成功的保证之一不等于这个人实际的价值人实际的价值人实际的价值人

> 说实话，我不太喜欢大部分的现代的歌，而我对古典音乐很感兴趣。在这篇作文中我会给您介绍特别喜欢的音乐
= 说实话，我不太喜欢大部分的现代的歌，但我对古典音乐很感兴趣。在这篇作文中，我会给您介绍我特别喜欢的音乐
< 说实话，我的身体非常爱。在这篇作文中，我会给您介绍我特别喜欢的音乐的目的。在这篇作文中，我会给您介绍我特别喜欢的音乐的

> 我毕业于新加坡国立大学设计系，学历可以在附件取得
= 我毕业于新加坡国立大学设计系，学历可以在附件查看
< 我毕业于新加坡国立大学，所以在附件查看，在法国在附件查看，学历可以在附件查看、美术馆、爬山、打猎等家务在一起，然